# 05 – KNMI weather

**Goal:** yearly weather per buurt (temperature, heating/cooling demand, sunlight).
**Inputs:** `All_Stations_Data.xlsx` (daily, 48 stations, 2019 – Sep 2026), raw KNMI `.txt` files (station coordinates only)
**Outputs:** `knmi_stations.csv`, `knmi_station_year.csv`, `buurt_knmi_station.csv`, `Data/processed/buurt_year_weather.csv`

## Cleaning
- Empty values were spaces (looked "100% filled") → converted to numbers; spaces became missing.
- Two note columns ("Closed", "210+265+311") dropped: one comment, not data.
- KNMI units: TG in 0.1 °C, Q in J/cm² → converted.
- Existing columns verified by recomputing (difference 0.0): `Hdd` = heating degree days base 18 °C, `Sdd` = cooling degree days base 18 °C, `solar_kwh_m2` = Q / 360.
- 13 of 48 stations measure wind only → not used. 35 have temperature, 33 sunlight.
- KNMI warns the series are inhomogeneous (not for climate trends) → fine for year-to-year variation.

## Yearly values
- Per station per year: mean temperature, HDD, CDD, sunlight (kWh/m²), frost days.
- Station-year accepted only with ≥ 330 days of data; 2026 incomplete (data until 20 Sep) → empty.
- Checks: 2021 coldest (HDD 2,828), 2022 sunniest, 2024 mild and dull.

## Linking to buurten
- Representative point per buurt (2023 shapes) → nearest station, separately for temperature and sunlight.
- Median distance 14.2 km, max 34.6 km. Main stations: Schiphol, Deelen, Berkhout, Lelystad, Leeuwarden.
- Spread in HDD between buurten within one year: ~12–14% → local weather adds information beyond the national trend.

## Timing (for the model)
- Liander SJV of year *t* is based on consumption in *t − 1* → join weather of **t − 1**.
- Forecast 2027 needs 2026 weather (incomplete) → use an average-weather assumption.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
KNMI = RAW / "3 knmi"

w = pd.read_excel(KNMI / "All_Stations_Data.xlsx")
w.columns = w.columns.str.strip()          # remove leading spaces like "   TG"
print("Shape:", w.shape)
print("Columns:", w.columns.tolist())

print("\nFirst rows:")
print(w.head(5).to_string())

print("\nDate range:", w["Data"].min(), "to", w["Data"].max())
print("Stations:", w["# STN"].nunique())

print("\n% filled per column:")
print((w.notna().mean() * 100).round(1).to_string())

print("\nValues in the two unnamed columns:")
print(w["Unnamed: 14"].value_counts(dropna=False).head(10).to_string())
print(w["Unnamed: 15"].value_counts(dropna=False).head(10).to_string())

Shape: (132652, 16)
Columns: ['# STN', 'Data', 'FG', 'TG', 'TN', 'TX', 'SQ', 'Q', 'RH', 'NG', 'UG', 'Hdd', 'Sdd', 'solar_kwh_m2', 'Unnamed: 14', 'Unnamed: 15']

First rows:
   # STN       Data   FG     TG     TN     TX     SQ      Q     RH     NG     UG   Hdd  Sdd  solar_kwh_m2 Unnamed: 14  Unnamed: 15
0    209 2019-01-01  108                                                           NaN  NaN           NaN         NaN          NaN
1    215 2019-01-01   72     77     58     89     13    161     10      7     79  10.3  0.0      0.447222      Closed  210+265+311
2    225 2019-01-01  112                                                           NaN  NaN           NaN         NaN          NaN
3    235 2019-01-01   78     80     60     96     22    249     -1      7     73  10.0  0.0      0.691667         NaN          NaN
4    240 2019-01-01   76     77     52     91     13    186      5      7     76  10.3  0.0      0.516667         NaN          NaN

Date range: 2019-01-01 00:00:00 to 2026-

In [2]:
w = w.drop(columns=["Unnamed: 14", "Unnamed: 15"]).rename(columns={"# STN": "stn", "Data": "date"})
w["date"] = pd.to_datetime(w["date"])

# spaces -> missing, text -> numbers
for c in ["FG", "TG", "TN", "TX", "SQ", "Q", "RH", "NG", "UG", "Hdd", "Sdd", "solar_kwh_m2"]:
    w[c] = pd.to_numeric(w[c].astype(str).str.strip(), errors="coerce")

# our own derived columns (KNMI units: TG in 0.1 degC, Q in J/cm2)
w["temp_mean_c"] = w["TG"] / 10
w["hdd"] = (18 - w["temp_mean_c"]).clip(lower=0)      # heating degree days, base 18
w["cdd"] = (w["temp_mean_c"] - 18).clip(lower=0)      # cooling degree days, base 18
w["solar_kwh_m2_calc"] = w["Q"] / 360                 # J/cm2 -> kWh/m2

# compare with the columns that were already in the file
print("Max difference hdd vs Hdd:            ", round((w["hdd"] - w["Hdd"]).abs().max(), 3))
print("Max difference cdd vs Sdd:            ", round((w["cdd"] - w["Sdd"]).abs().max(), 3))
print("Max difference solar calc vs original:", round((w["solar_kwh_m2_calc"] - w["solar_kwh_m2"]).abs().max(), 3))

# which stations measure temperature and sunlight?
per_stn = w.groupby("stn").agg(
    days=("date", "size"),
    temp_pct=("TG", lambda s: s.notna().mean() * 100),
    sun_pct=("Q", lambda s: s.notna().mean() * 100),
).round(1)
print("\nStations with temperature data:", (per_stn["temp_pct"] > 90).sum(), "of", len(per_stn))
print("Stations with sunlight data:   ", (per_stn["sun_pct"] > 90).sum(), "of", len(per_stn))
print("\nStations WITHOUT (enough) temperature data:")
print(per_stn[per_stn["temp_pct"] <= 90].to_string())

Max difference hdd vs Hdd:             0.0
Max difference cdd vs Sdd:             0.0
Max difference solar calc vs original: 0.0

Stations with temperature data: 35 of 48
Stations with sunlight data:    33 of 48

Stations WITHOUT (enough) temperature data:
     days  temp_pct  sun_pct
stn                         
209  2820       0.0      0.0
225  2820       0.0      0.0
248  2820       0.0      0.0
258  2820       0.0      0.0
285  2820       0.0      0.0
308  2820       0.0      0.0
312  2820       0.0      0.0
313  2820       0.0      0.0
315  2820       0.0      0.0
316  2820       0.0      0.0
324  2820       0.0      0.0
331  2820       0.0      0.0
343  2820       0.0      0.0


In [3]:
txt_files = sorted((KNMI / "Stations Data Raw").glob("*.txt"))
print("Raw files:", len(txt_files))

with open(txt_files[0], encoding="latin-1") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 30:
            break

Raw files: 15
# Opmerking: door stationsverplaatsingen en veranderingen in waarneemmethodieken zijn deze tijdreeksen van uurwaarden mogelijk inhomogeen! Dat betekent dat deze reeks van gemeten waarden niet geschikt is voor trendanalyse. Voor studies naar klimaatverandering verwijzen we naar de gehomogeniseerde dagreeksen <http://www.knmi.nl/nederland-nu/klimatologie/daggegevens> of de Centraal Nederland Temperatuur <http://www.knmi.nl/kennis-en-datacentrum/achtergrond/centraal-nederland-temperatuur-cnt>.
#
# SOURCE: ROYAL NETHERLANDS METEOROLOGICAL INSTITUTE (KNMI)
# Comment: These time series are inhomogeneous because of station relocations and changes in observation techniques. As a result these series are not suitable for trend analysis. For climate change studies we refer to the homogenized series of daily data <http://www.knmi.nl/nederland-nu/klimatologie/daggegevens> or the Central Netherlands Temperature <http://www.knmi.nl/kennis-en-datacentrum/achtergrond/centraal-nederland-te

In [4]:
import re

pattern = re.compile(r"^#\s+(\d{3})\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s+(.+)$")

rows = []
for f in txt_files:
    with open(f, encoding="latin-1") as file:
        for line in file:
            if not line.startswith("#"):
                break                       # header finished, data starts
            m = pattern.match(line.strip())
            if m:
                rows.append({
                    "stn": int(m.group(1)),
                    "lon": float(m.group(2)),
                    "lat": float(m.group(3)),
                    "alt_m": float(m.group(4)),
                    "name": m.group(5).strip(),
                })

stations = pd.DataFrame(rows).drop_duplicates("stn").sort_values("stn")
print("Stations with coordinates:", len(stations))

# add which stations measure temperature and sunlight
stations = stations.merge(per_stn[["temp_pct", "sun_pct"]], left_on="stn", right_index=True, how="left")
stations["has_temp"] = stations["temp_pct"] > 90
stations["has_sun"] = stations["sun_pct"] > 90

print("\nStations in the weather data but WITHOUT coordinates:",
      sorted(set(per_stn.index) - set(stations["stn"])))
print("\nStations with temperature:")
print(stations[stations["has_temp"]][["stn", "name", "lon", "lat", "has_sun"]].to_string(index=False))

stations.to_csv(PROCESSED / "knmi_stations.csv", index=False)
print("\nSaved: knmi_stations.csv")

Stations with coordinates: 51

Stations in the weather data but WITHOUT coordinates: []

Stations with temperature:
 stn                    name      lon       lat  has_sun
 215             Voorschoten 4.437000 52.141000     True
 235         De Kooy Airport 4.781000 52.928000     True
 240        Schiphol Airport 4.790000 52.318000     True
 242       Vlieland Vliehors 4.921000 53.241000    False
 249                Berkhout 4.979000 52.644000     True
 251      Hoorn Terschelling 5.346000 53.392000     True
 257            Wijk aan Zee 4.603000 52.506000     True
 260                 De Bilt 5.180000 52.100000     True
 267                Stavoren 5.384000 52.898000     True
 269        Lelystad Airport 5.520000 52.458000     True
 270      Leeuwarden Airport 5.752000 53.224000     True
 273               Marknesse 5.888000 52.703000     True
 275          Deelen Airport 5.873000 52.056000     True
 277              Lauwersoog 6.200000 53.413000     True
 278                   Heino 

In [5]:
w["year"] = w["date"].dt.year

yearly = w.groupby(["stn", "year"]).agg(
    days=("date", "size"),
    temp_days=("TG", "count"),        # count = number of non-missing values
    sun_days=("Q", "count"),
    temp_mean_c=("temp_mean_c", "mean"),
    hdd_year=("hdd", "sum"),
    cdd_year=("cdd", "sum"),
    solar_kwh_m2_year=("solar_kwh_m2_calc", "sum"),
    frost_days=("TN", lambda s: (s < 0).sum()),
).reset_index()

# only trust complete years
MIN_DAYS = 330
temp_cols = ["temp_mean_c", "hdd_year", "cdd_year", "frost_days"]
yearly.loc[yearly["temp_days"] < MIN_DAYS, temp_cols] = np.nan
yearly.loc[yearly["sun_days"] < MIN_DAYS, "solar_kwh_m2_year"] = np.nan

# national overview: average over stations with complete data
overview = yearly.groupby("year").agg(
    stations_complete=("hdd_year", "count"),
    temp_mean_c=("temp_mean_c", "mean"),
    hdd_year=("hdd_year", "mean"),
    cdd_year=("cdd_year", "mean"),
    solar_kwh_m2_year=("solar_kwh_m2_year", "mean"),
    frost_days=("frost_days", "mean"),
).round(1)
print(overview.to_string())

yearly.to_csv(PROCESSED / "knmi_station_year.csv", index=False)
print("\nSaved: knmi_station_year.csv", yearly.shape)

      stations_complete  temp_mean_c  hdd_year  cdd_year  solar_kwh_m2_year  frost_days
year                                                                                   
2019                 34         11.1    2640.8     135.2             1109.1        37.9
2020                 34         11.6    2489.9     137.7             1145.9        23.4
2021                 34         10.4    2828.0      55.9             1067.0        48.9
2022                 34         11.6    2492.9     150.9             1204.4        35.7
2023                 34         11.7    2442.5     139.5             1119.3        36.5
2024                 34         11.7    2413.1     120.0             1053.3        19.9
2025                 33         11.3    2553.6     119.9             1176.8        50.5
2026                  0          NaN       NaN       NaN                NaN         NaN

Saved: knmi_station_year.csv (377, 10)


In [6]:
import geopandas as gpd

# buurten we need (Liander area, 2023 codes) -> one point inside each buurt
buurten = gpd.read_file(RAW / "External" / "cbs_buurten_2023_simplified.geojson")
need = set(pd.read_csv(PROCESSED / "buurt_year_electricity.csv")["buurtcode"])
b = buurten[buurten["buurtcode"].isin(need)][["buurtcode", "geometry"]].to_crs("EPSG:28992")
b["geometry"] = b.representative_point()
print("Buurten:", len(b))

# stations as points (Dutch grid in metres)
st = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations["lon"], stations["lat"]),
                      crs="EPSG:4326").to_crs("EPSG:28992")
st_temp = st[st["has_temp"]][["stn", "name", "geometry"]]
st_sun = st[st["has_sun"]][["stn", "geometry"]]

# nearest station for temperature and for sunlight
bt = gpd.sjoin_nearest(b, st_temp, how="left", distance_col="dist_temp_m").drop_duplicates("buurtcode")
bs = gpd.sjoin_nearest(b, st_sun, how="left", distance_col="dist_sun_m").drop_duplicates("buurtcode")

link = pd.DataFrame({
    "buurtcode": bt["buurtcode"],
    "stn_temp": bt["stn"],
    "stn_temp_name": bt["name"],
    "dist_temp_km": bt["dist_temp_m"] / 1000,
}).merge(pd.DataFrame({
    "buurtcode": bs["buurtcode"],
    "stn_sun": bs["stn"],
    "dist_sun_km": bs["dist_sun_m"] / 1000,
}), on="buurtcode")

print("\nDistance to nearest temperature station (km):")
print(link["dist_temp_km"].describe().round(1).to_string())
print("\nMost used temperature stations:")
print(link["stn_temp_name"].value_counts().head(10).to_string())

link.to_csv(PROCESSED / "buurt_knmi_station.csv", index=False)
print("\nSaved: buurt_knmi_station.csv")

Buurten: 5672

Distance to nearest temperature station (km):
count    5672.0
mean       14.4
std         6.9
min         0.1
25%         8.9
50%        14.2
75%        19.4
max        34.6

Most used temperature stations:
stn_temp_name
Schiphol Airport      918
Deelen Airport        836
Berkhout              551
Lelystad Airport      486
Leeuwarden Airport    463
Voorschoten           417
Wijk aan Zee          316
Hupsel                295
Herwijnen             204
De Bilt               201

Saved: buurt_knmi_station.csv


In [7]:
wcols = ["temp_mean_c", "hdd_year", "cdd_year", "frost_days", "solar_kwh_m2_year"]

yt = yearly[["stn", "year", "temp_mean_c", "hdd_year", "cdd_year", "frost_days"]].rename(columns={"stn": "stn_temp"})
ys = yearly[["stn", "year", "solar_kwh_m2_year"]].rename(columns={"stn": "stn_sun"})

# every buurt x every weather year
years = pd.DataFrame({"year": sorted(yearly["year"].unique())})
bw = link[["buurtcode", "stn_temp", "stn_sun"]].merge(years, how="cross")
bw = bw.merge(yt, on=["stn_temp", "year"], how="left").merge(ys, on=["stn_sun", "year"], how="left")

# gaps (incomplete station-years) -> national average of that year, and flag it
bw["weather_filled"] = bw[wcols].isna().any(axis=1)
national = yearly.groupby("year")[wcols].mean()
for c in wcols:
    bw[c] = bw[c].fillna(bw["year"].map(national[c]))

print("Shape:", bw.shape)
print("\n% of buurten with filled (national average) weather, per year:")
print((bw.groupby("year")["weather_filled"].mean() * 100).round(1).to_string())
print("\nHeating degree days per year (spread between buurten):")
print(bw.groupby("year")["hdd_year"].describe()[["mean", "min", "max"]].round(0).to_string())

bw.to_csv(PROCESSED / "buurt_year_weather.csv", index=False)
print("\nSaved: buurt_year_weather.csv")

Shape: (45376, 10)

% of buurten with filled (national average) weather, per year:
year
2019      0.0
2020      0.0
2021      0.0
2022      0.0
2023      0.0
2024      0.0
2025      0.0
2026    100.0

Heating degree days per year (spread between buurten):
        mean     min     max
year                        
2019  2667.0  2512.0  2821.0
2020  2525.0  2363.0  2733.0
2021  2860.0  2685.0  3044.0
2022  2526.0  2366.0  2758.0
2023  2469.0  2306.0  2683.0
2024  2432.0  2300.0  2594.0
2025  2584.0  2408.0  2808.0
2026     NaN     NaN     NaN

Saved: buurt_year_weather.csv
